# 2. 에이전트 RAG


- 에이전트 RAG는 RAG에 LLM의 의사결정 능력을 결합한 시스템입니다.
- ReAct 방법론은 대표적인 에이전트 구현 방식 중 하나입니다.
- 2개의 서로 다른 PDF파일로부터 2개의 검색기를 만들어 ReAct 에이전트와 연결하여 복잡한 질문을 처리할 수 있는 에이전트 RAG를 구현해보겠습니다.


In [1]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain.tools.retriever import create_retriever_tool
from langchain import hub
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

- PyMuPDFLoader: PDF 문서를 읽어들이고 텍스트를 추출하는 도구입니다.
- RecursiveCharacterTextSplitter: 긴 문서를 의미 있는 단위로 분할하는 도구로, 문장과 단락의 문맥을 보존하며 텍스트를 청크 단위로 나눕니다.
- OllamaEmbeddings: 임베딩 모델을 사용해 텍스트를 벡터로 변환합니다.
- Chroma: 벡터화된 텍스트를 저장하고 검색하기 위한 벡터 데이터베이스입니다.
- create_retriever_tool: 벡터 검색을 ReAct에이전트의 도구로 변환합니다.
- hub: LangChain의 프롬프트 템플릿 저장소에 접근합니다.
- ChatOpenAI: 오픈AI의 챗GPT 모델을 활용하기 위한 인터페이스입니다.
- AgentExecutor, create_react_agent: ReAct 에이전트를 생성하고 실행하는 핵심 컴포넌트입니다.
- PromptTemplate: 에이전트의 프롬프트를 템플릿화하여 관리합니다.


#


# 에이전트 도구 만들기


In [2]:
embd = OllamaEmbeddings(model="bge-m3")

- PDF 문서를 벡터 데이터베이스로 변환하고 사용자 질의로부터 유사한 문서를 반환하는 검색기 객체인 retriever를 생성하는 함수 create_pdf_retriever를 구현합니다.


In [3]:
def create_pdf_retriever(
    pdf_path: str,  # PDF 파일 경로
    persist_directory: str,  # 벡터 스토어 저장 경로
    embedding_model: OllamaEmbeddings,  # 임베딩 모델
    chunk_size: int = 512,  # 텍스트 청크 크기
    chunk_overlap: int = 0,  # 청크 오버랩 크기
) -> Chroma.as_retriever:
    # PDF 파일 로드
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    # 청킹
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    doc_splits = text_splitter.split_documents(data)

    # 벡터 스토어로 적재
    vectorstore = Chroma.from_documents(
        persist_directory=persist_directory,
        documents=doc_splits,
        embedding=embedding_model,
    )

    return vectorstore.as_retriever()

- 이 함수를 이용하여 일본 ICT 정책에 대해 검색하는 검색기와 미국 ICT 정책에 대해 검색하는 검색기를 각각 만들어 봅시다.


In [4]:
# 일본 ICT 정책 데이터베이스 생성
retriever_japan = create_pdf_retriever(
    pdf_path="ict_japan_2024.pdf",
    persist_directory="db_ict_policy_japan_2024",
    embedding_model=embd,
)

# 미국 ICT 정책 데이터베이스 생성
retriever_usa = create_pdf_retriever(
    pdf_path="ict_usa_2024.pdf",
    persist_directory="db_ict_policy_usa_2024",
    embedding_model=embd,
)

- 같은 경로를 사용한다면 두번째 문서를 처리할 때 첫번째 문서의 데이터가 덮어써지거나 섞일 수 있기 때문에 경로를 분리
- create_retriever_tool 함수를 하용해 ReAct 에이전트가 사용할 수 있는 검색 도구로 변환


In [5]:
jp_engine = create_retriever_tool(
    retriever=retriever_japan,
    name="japan_ict",
    description="일본의 ICT 시장 동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)
usa_engine = create_retriever_tool(
    retriever=retriever_usa,
    name="usa_ict",
    description="미국의 ICT 시장 동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

tools = [jp_engine, usa_engine]

- create_retriever_tool은 retriever 객체를 ReAct 에이전트가 활용할 수 있는 도구형태로 변환해주는 함수입니다.
- description에는 각 검색기의 상세한 용도를 작성해야 합니다.


# 에이전트 프롬프트 설정

- 랭체인에서 ReAct 에이전트를 동작시킬 때 기본값으로 제공하는 프롬프트가 있습니다.
- 너무 단순합니다. 제대로 된 동작을 유도하려면 사용자가 조금 더 자세하게 작성하는 것이 좋습니다.


In [6]:
# 랭체인 기본 제공 프롬프트
prompt_react = hub.pull("hwchase17/react")
print(prompt_react)
print("--프롬프트 끝--")

c:\workspace\python\rag_master\.venv\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'
--프롬프트 끝--


주어진 질문들에 대해 최선을 다해 답변하세요. 다음과 같은 도구들을 사용할 수 있습니다:  
{tools}  
다음 형식을 사용하세요:  
Question: 답변해야 할 입력 질문  
Thought: 무엇을 해야 할지 항상 고민해야 합니다.  
Action: 수행할 행동(반드시 [{tool_names}] 중 하나여야 함)  
Action Input: 행동에 필요한 입력값  
Observation: 행동의 결과  
...(이 Thought/Action/Action Input/Observation 과정은 N번 반복될 수 있습니다.)  
Thought: 이제 최종 답을 알았습니다.  
Final Answer: 원래 입력 질문에 대한 최종 답변  
시작!  
Quesion: {input}  
Thought: {agent_scratchpad}


- 프롬프트 사이에 중괄호 {}로 감싼 부분은 변수에 해당합니다.
- 실제 실행 시에는 각 변수에 적절한 값이 채워지는 구조입니다.


- {tools}:
  - 에이전트가 사용할 수 있는 도구들의 설명이 포함된 목록입니다.
  - create_retriever_tools()에서 정의한 도구들의 이름과 설명이 해당위치에 들어갑니다.
- {tool_names}:
  - 에이전트가 선택할 수 있는 도구의 이름들을 기재합니다.
  - 여기에는 설명없이 도구들의 이름만 리스트 형태로 들어갑니다.
- {input}:
  - 사용자가 현재 물어본 질문이 이 부분에 들어갑니다.
- {agent_scratchpad}:
  - 에이전트의 모든 사이클(Throught/Action/Observation의 기록)이 이 부분에 누적됩니다.
  - 이를 통해 에이전트는 이전의 사이클들을 참고하여 다음 행동을 결정할 수 있습니다.


- 먼저 에이전트가 사용할 수 있는 도구들을 설명하고, 문제 해결을 위해 "Throught/Action/Action Input/Observation"의 사이클을 N번 반복할 수 있다고 안내합니다.
- 주목할 점은 Throught 단계를 각 Action 전후에 하도록 지시한다는 점입니다.
- 이는 에이전트가 행동을 취하기 전에 충분히 고민하고, 또 행동의 결과를 관찰한 후에도 다시 한번 생각하면서 사이클을 돌도록 유도합니다.
- 사용자의 Question에 답하기 위한 Observation이 충분히 취합되면, Final Answer전에 이제 최종 답을 알았습니다. 라는 명시적인 마지막 생각 단계를 작성 하도록 하여 에이전트가 자신의 결론에 확신을 갖는 경우에 답변하도록 설계되었습니다.


In [17]:
template = """다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

{tools}

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요.
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다.
Final Answer: 원래 입력된 질문에 대한 최종 답변

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input format] 이 사이클의 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- **동일한 Observation이 반복되거나, 더 이상 새로운 정보를 얻을 수 없다고 판단되면 즉시 "Thought: 이제 최종 답변을 알겠습니다."로 넘어가세요.**
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것이 중요합니다.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

- 기본으로 제공되는 프롬프트와 달리 ## 추가적인 주의사항이라는 내용이 추가되었습니다.
- 더 나은 성능을 얻기위해 추가한 내용입니다.


# 에이전트 객체 생성


In [18]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

react_agent = create_react_agent(llm, tools=tools, prompt=prompt)

react_agent_executor = AgentExecutor(
    agent=react_agent, tools=tools, verbose=True, handle_parsing_errors=True
)

- temperature 값을 0으로 사용하여 모델의 답변 다양성을 줄이는 이유는 예측하지 못한 창의적 답변보다는 지시사항에 명확히 따르기를 기대하기 때문입니다.


# 에이전트 RAG 실습


In [19]:
result = react_agent_executor.invoke({"input": "일본과 미국의 ICT 기관 협력 사례"})



> Entering new AgentExecutor chain...
일본과 미국의 ICT 기관 협력 사례에 대한 정보를 수집해야 합니다. 이 주제는 두 나라의 ICT 관련 기관들이 어떻게 협력하고 있는지를 다루고 있으므로, 일본과 미국의 ICT 시장 동향을 각각 조사하여 관련 사례를 찾는 것이 좋겠습니다. 먼저 일본의 ICT 기관 협력 사례를 알아보겠습니다.

Action: japan_ict  
Action Input: '일본 ICT 기관 협력 사례'  16 
Ⅰ. ICT 국가 산업 현황
 6. 한국 협력 및 국내기업 진출사례
 한국-일본 FTA 체결 여부
• 한국과 일본은 2010년 1월 1일 포괄적 경제동반자협정(CEPA)이 발효됨
• 2022년 1월 한국-일본 CEPA 개선 협상이 2년 반 만에 재개됨. 해당 협상에서는 포스트 코로나 
시대를 대비해 공급망 강화, 기후변화, 백신, 디지털 협력 등의 이슈 관련 양국 협력 확대 
방안을 논의해 나가기로 합의함
 한국-일본 ICT 기관 협력 사례

Ⅰ ICT 국가산업현황 
 4
(*) SUMMARY
1. 국가 개황
2. ICT 정부기구
3. ICT 주요정책
4. ICT 주요법령및규제
5. ICT 주요기업
6. 한국 협력 및 국내기업 진출사례
Ⅱ ICT 이슈Top 10
 16
(*) SUMMARY
① 일본, 아시아에서 두 번째로 큰 데이터센터 허브
② 일본, 자체 개발 소프트웨어로 사이버보안 강화
③ 일본, Web3 산업 성장 촉진 도모
④ 일본, 정부 행정 업무에 생성형 AI 도입
⑤ 일본, 첫 자체 제작 양자컴퓨터 공개
⑥ 일본, 6G 기술 강화 위해 협력 및 규제 완화

• 2023년 10월 이종호 과학기술정보통신부 장관이 일본을 방문해 총무성, 문부과학성과 장관회담 
진행하고 한일 간 공동연구 및 인력교류 활성화 방안 논의
 한국-일본 ICT 기업 진출 사례
• 전자, 디지털 치료제, 무인 로봇, 클라우드 등 다양한 한국 ICT 사업의 일본 진출이 활발함
[표 11] 한국-일본 협력 현황
구분
날짜


In [20]:
print("최종 답변:", result["output"])

최종 답변: 일본과 미국의 ICT 기관 협력 사례는 다음과 같습니다.

1. **일본의 ICT 기관 협력 사례**:
   - 일본의 총무성(Ministry of Internal Affairs and Communications, MIC)은 정보통신 정책을 기획하고 추진하는 중앙행정기관입니다. 최근 이종호 과학기술정보통신부 장관이 일본을 방문하여 총무성과 문부과학성과의 장관 회담을 진행하였고, 한일 간 공동 연구 및 인력 교류 활성화 방안을 논의했습니다. 이러한 협력은 포스트 코로나 시대를 대비한 공급망 강화와 디지털 협력 확대를 목표로 하고 있습니다.

2. **미국의 ICT 기관 협력 사례**:
   - 한국과 미국 간의 ICT 협력은 과학기술정보통신부와 미국 기관 간의 협력으로 주목받고 있습니다. 예를 들어, 미국은 일본과의 양자컴퓨팅 개발 협력에 힘쓰고 있으며, 사이버 보안 대응을 강화하기 위한 다양한 노력을 기울이고 있습니다. 또한, 미국의 여러 기업과 정부기관은 사이버 공격에 대한 대응을 강화하고 있으며, 이는 ICT 분야의 협력과 기술 교류를 더욱 촉진하고 있습니다.

이러한 사례들은 일본과 미국이 ICT 분야에서 어떻게 협력하고 있는지를 보여주며, 양국 간의 기술적 교류와 공동 연구가 활발히 이루어지고 있음을 나타냅니다.


- 에이전트는 첫번째 생각에서 일본의 ICT 기관 협력 사례를 알아보기 위해 자료를 검색합니다.
- japan_ict를 사용해야 한다고 판단한 것입니다.
- 다음 생각에서 질문을 해결하기에 충분한 정보를 얻었다고 판단하고 최종 답변을 작성했습니다.


In [21]:
result = react_agent_executor.invoke(
    {"input": "미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해줘."}
)



> Entering new AgentExecutor chain...
미국과 일본의 ICT 주요 정책에 대한 공통점과 차이점을 이해하기 위해서는 두 나라의 ICT 정책에 대한 정보를 수집해야 합니다. 미국의 ICT 정책에 대한 정보를 먼저 검색해 보겠습니다.

Action: usa_ict  
Action Input: '미국 ICT 주요 정책'  9 
Ⅰ. ICT 국가 산업 현황
 3. ICT 주요정책
  ① 국가 스펙트럼 전략(National Spectrum Strategy)
 미국 신규 주파수 공급 전략
• 2023년 11월 조 바이든(Joe Biden) 행정부는 2,700MHz 이상의 신규 주파수 공급 전략인 
‘국가 스펙트럼 전략(National Spectrum Strategy)’을 발표함. 또 명확한 스펙트럼 정책의 제시 
및 스펙트럼 관련 갈등 해결 프로세스 확립을 위한 ‘미국 스펙트럼 정책 현대화에 관한 대통령 
각서(Presidential Memorandum on modernizing U.S. spectrum policy)’를 발표함
• 미국 정부는 스펙트럼 정책에 대한 이하의 비전을 채택함

Ⅰ ICT 국가산업현황 
 4
(*) SUMMARY
1. 국가 개황
2. ICT 정부기구
3. ICT 주요정책
4. ICT 주요법령및규제
5. ICT 주요기업
6. 한국 협력 및 국내기업 진출사례
Ⅱ ICT 이슈Top 10
 16
(*) SUMMARY
① 미국 빅테크 기업, 인공지능 챗봇 개발에 주력
② 미국, 일본과 양자컴퓨팅 개발 협력
③ 미국, 우주 클라우드 컴퓨팅 시장 주도
④ 미국, 드론 배송 도입 활발
⑤ 미국, 긍정적인 의료 AI 인식 바탕으로 연구 활발
⑥ 미국, 반도체 산업 활성화에 박차

13 
Ⅰ. ICT 국가 산업 현황
 4. ICT 주요법령 및 규제
  ② 반도체 과학법(CHIPS and Science Act)
 반도체·전자 기업, $1,660억 규모 투자 유치 
• 조 바이든(Joe Biden) 미국 대통령은 2022년 7월 ‘반도

In [22]:
print("최종 답변:", result["output"])

최종 답변: 미국과 일본의 ICT 주요 정책은 반도체 산업과 AI 기술의 중요성을 공통적으로 인식하고 있으며, 이를 통해 산업 경쟁력을 강화하려고 하고 있습니다. 그러나 미국은 스펙트럼 정책과 같은 통신 인프라에 대한 구체적인 전략을 강조하는 반면, 일본은 산업 경쟁력 강화를 위한 법률 개정과 생성형 AI 사용 지침을 통해 규제 측면에서의 접근을 강조하고 있습니다.


- 먼저 usa_ict를 호출하여 미국의 ICT 정책에 대한 정보를 얻습니다.
- 두번째 생각에서 일본의 ICT 주요 정책에 대한 정보를 얻습니다.
- 에이전트는 질문을 해결하기에 충분한 정보를 얻었다고 판단하고 최종 답변을 작성합니다.


- 보다 복잡한 질문을 입력해봅니다.


In [23]:
result = react_agent_executor.invoke(
    {
        "input": "미국의 ICT 관련 정부 기구, 주요 법령, 국내 기업 진출 사례 각각 따로 검색해. 그렇게 해서 정보 좀 모아봐. 그리고 나서 일본의 AI 정책도 알려줘"
    }
)



> Entering new AgentExecutor chain...
Question: 미국의 ICT 관련 정부 기구, 주요 법령, 국내 기업 진출 사례 각각 따로 검색해. 그렇게 해서 정보 좀 모아봐. 그리고 나서 일본의 AI 정책도 알려줘  
Thought: 미국의 ICT 관련 정부 기구, 주요 법령, 그리고 국내 기업 진출 사례를 각각 따로 검색해야 하므로, 세 번의 검색을 진행해야 한다. 첫 번째로 미국의 ICT 관련 정부 기구에 대한 정보를 검색하겠다.  
Action: usa_ict  
Action Input: "미국의 ICT 관련 정부 기구"  8 
Ⅰ. ICT 국가 산업 현황
 2. ICT 정부기구
 ③ 국가과학기술위원회(NSTC) 
 미국 국가과학기술위원회(NSTC)
• 국가과학기술위원회(National Science and Technology Council)는 1993년 설립되었으며 
백악관 산하의 자문기구로 역할을 담당하고 있음
• 국가과학기술위원회의 기능은 ① 과학기술정책 결정과정의 조정 ② 과학기술정책결정 프로그램의 
목표 지원 ③ 연방정부 전반에 대한 과학기술정책 의제 통합에 기여 ④ 과학기술 분야의 연방

7 
Ⅰ. ICT 국가 산업 현황
 2. ICT 정부기구
  ② 연방통신위원회(FCC)
 미국 연방통신위원회(FCC)
• 미국 연방통신위원회(Federal Communications Commission)는 1934년에 제정된 연방통신법 
(Communications Act of 1934)에 의해 설립된 연방정부의 정보통신(ICT)･방송 독립 행정 기관임
• 주요 업무는 정보통신기술(ICT)･방송에 관한 기본계획 수립 등 정책 총괄, 소유제한 등 시장 규제 및 
경쟁정책 수립, 정보통신기술(ICT)･방송 사업자 허가 및 감독, 주파수 배분 및 할당 관리, 방송편성

6 
Ⅰ. ICT 국가 산업 현황
 2. ICT 정부기구
  ① 국가통신정보관리청(NTIA)
 국가통신정보관리청(NTIA)
• 
상무부 
산하의 
국가통신정보관리청(Na

In [24]:
print("최종 답변:", result["output"])

최종 답변: **미국의 ICT 관련 정보:**

1. **정부 기구:**
   - **국가통신정보관리청(NTIA)**: 상무부 산하의 행정 기관으로, 통신 및 정보 정책 문제에 대해 대통령에게 자문을 제공.
   - **연방통신위원회(FCC)**: 1934년에 제정된 연방통신법에 의해 설립된 독립 행정 기관으로, ICT 및 방송 정책을 총괄.
   - **국가과학기술위원회(NSTC)**: 백악관 산하의 자문기구로, 과학기술 정책 결정과정의 조정 및 연방정부의 과학기술 정책 의제 통합에 기여.

2. **주요 법령:**
   - **AI 행정 명령**: 2023년 11월 발표된 행정명령으로, AI 관련 위험을 관리하기 위한 새로운 표준을 수립.
   - **반도체 과학법(CHIPS and Science Act)**: 2022년 7월 승인된 법로, 반도체 및 전자 기업에 대한 대규모 투자 유치를 목표로 하며, 미국의 경쟁력과 공급망을 강화하는 데 기여.

3. **국내 기업 진출 사례:**
   - 한국과 미국 간의 FTA 체결과 ICT 협력 사례가 주목되며, 한국 기업들이 미국 시장에 진출하는 데 있어 긍정적인 영향을 미치고 있다.

**일본의 AI 정책:**
- 일본 내각부는 AI 모델 성능 향상을 위해 정부가 보유한 안전･안심 데이터를 AI 개발자에게 AI 학습 데이터로 제공하는 액션플랜을 공개하였다. 이 정책은 AI 및 데이터 분석 전문 인력이 부족한 일본에서 AI의 적절한 활용을 장려하고 기업의 디지털 전환을 지원하는 것을 목표로 하고 있다. 또한, 일본 개인정보보호위원회는 생성형 AI 서비스 사용에 관한 지침을 발표하여 기업과 일반 사용자가 유의해야 할 점을 상세히 제시하고 있다.


- 시작과 함께 에이전트는 질문의 의도에 따라 적절한 도구를 배정하기 위해 생각 과정을 거칩니다.
- 미국 관련 세가지 정보를 usa_ict 도구로 순차적으로 검색합니다.
- 각 검색마다 관찰 결과를 확인하고 다음 검색으로 넘어갑니다.
- 미국 관련 정보 수집을 마친 후, japan_ict 도구를 사용해 일본 AI 정책을 검색합니다.
- 모든 정보를 수집하여 최종 답변을 작성합니다.
